In [ ]:
import pandas as pd

### Notes on the files
There are 2 sets of files: from MIMIC and from SNOMED. 
Processed files are in Archive folder
MIMIC files are prefixed with 'mimic_'

In [ ]:
admissions = pd.read_csv('admissions.csv')
admissions

In [ ]:
cols_to_drop = ['admit_provider_id',
                'admission_type',
                'admission_location',
                'discharge_location',
                'insurance',
                'marital_status',
                'race',
                'edregtime',
                'edouttime',
                'hospital_expire_flag',
               'language',
               'deathtime']
admissions_n4j = admissions.drop(columns=cols_to_drop)
admissions_n4j

In [ ]:
admissions_n4j.to_csv('mimic_admissions.csv', index=False)

In [ ]:
dx = pd.read_csv('diagnoses_icd.csv')
dx

In [ ]:
# there are both ICD9 and ICD10 codings of the diagnoses 
icd = dx['icd_version'].unique()
icd

In [ ]:
icd9 = dx[dx['icd_version']==9]
icd9

In [ ]:
# the icd code has no separators, we need to add in. First check that all entries are at least 3 digits long
# ge means "greater than or equal to"
is_long_enough = icd9['icd_code'].str.len().ge(3).all()
is_long_enough

In [ ]:
# insert the separator.
def fix_icd9(code):
    if len(code)==3:
        return code
    else:
        return code[:3] + '.' + code[3:]

icd9['ICD9'] = icd9['icd_code'].apply(fix_icd9)
icd9

The ICD9CM to SNOMED mapping table can be downloaded from https://www.nlm.nih.gov/research/umls/mapping_projects/icd9cm_to_snomedct.html 
It comes in 2 tables: a 1-to-1 table, and a 1-to-many table.
We'll concatenate the 2 tables after extracting our 2 attributes of interest: ICD_CODE and SNOMED_CID.

In [ ]:
# the encoding from UMLS ICD9CM_to_SNOMED conversion table is in cp1252
# change dtype to string for easier processing later (numerics get auto-converted to scientific notation)
icd9_umls1=pd.read_csv('icd9_snomed_TO1.txt', encoding='cp1252', dtype=str, sep='\t')
icd9_umls1

In [ ]:
# see what unique values in some of the columns
unique_icd9 = {}

icd9_umls_explore = icd9_umls1.drop(columns = ['ICD_CODE', 'ICD_NAME', 'IP_USAGE', 'OP_USAGE', 'AVG_USAGE', 'SNOMED_CID', 'SNOMED_FSN', 'CORE_USAGE'])

for col in icd9_umls_explore:
   unique_icd9.update({col: list(icd9_umls_explore[col].unique())})
    

In [ ]:
unique_icd9

In [ ]:
# keep just the code mappings
icd9_snomed_1 = icd9_umls1[['ICD_CODE', 'SNOMED_CID']]
icd9_snomed_1

In [ ]:
# do the same for the 1-to-many file
icd9_umls2 = pd.read_csv('icd9_snomed_TOM.txt', encoding='cp1252', dtype=str, sep='\t')
icd9_umls2

In [ ]:
# drop NaN rows for SNOMED_CID
icd9_umls2 = icd9_umls2.dropna(subset=['SNOMED_CID'])
icd9_umls2

In [ ]:
icd9_snomed_2 = icd9_umls2[['ICD_CODE', 'SNOMED_CID']]
icd9_snomed_2

In [ ]:
# concatenate the 2 sets
icd9_snomed = pd.concat([icd9_snomed_1, icd9_snomed_2], ignore_index=True, axis=0)
icd9_snomed

In [ ]:
# check if duplicates present
snomed_duplicates_present = icd9_snomed['SNOMED_CID'].duplicated().any()
snomed_duplicates_present

In [ ]:
# see the duplicates
counts = icd9_snomed['SNOMED_CID'].value_counts()
counts

In [ ]:
# the mapping table is a many-to-many table. 
# Since we don't know which SNOMED codes are relevant to the patients, we'll keep all of them.

# create list where each ICD9 points to a list of SNOMED codes.
mapping_list = icd9_snomed.groupby('ICD_CODE')['SNOMED_CID'].apply(list)
mapping_list

In [ ]:
# create a column containing the mapped snomed lists
icd9['snomed'] = icd9['ICD9'].map(mapping_list)
icd9

In [ ]:
# then explode the snomed lists
icd9_expanded = icd9.explode('snomed')
icd9_expanded

In [ ]:
# now do the same for the icd10
icd10 = dx[dx['icd_version']==10]
icd10

In [ ]:
# again there is no punctuation in the ICD10 codes. 
# First check that all entries are at least 3 digits long
# ge means "greater than or equal to"
is_long_enough = icd10['icd_code'].str.len().ge(3).all()
is_long_enough


In [ ]:
# we'll reuse the fix_icd9 function
# but we'll keep the ICD9 name in the fixed column for easier merging later
icd10['ICD9'] = icd10['icd_code'].apply(fix_icd9)
icd10

The SNOMED to ICD10 mapping table can be downloaded from https://www.nlm.nih.gov/healthit/snomedct/us_edition.html 
we'll open the .tsv file (which I have renamed as icd10_snomed here.

In [ ]:
# open the file with python engine to avoid c-engine regex handling of tab separators
icd10_snomed = pd.read_csv('icd10_snomed.tsv', engine='python', dtype=str, sep='\\t')
icd10_snomed

In [ ]:
for i, col in enumerate(icd10_snomed.columns):
    print(i, col)

In [ ]:
# the 2 attributes of interest are ReferencedComponentId and MapTarget
icd10_umls_snomed = icd10_snomed[['mapTarget','referencedComponentId']]
icd10_umls_snomed = icd10_umls_snomed.rename(columns = {'mapTarget': 'ICD_CODE', 'referencedComponentId': 'SNOMED_CID'})

In [ ]:
icd10_umls_snomed

In [ ]:
# now repeat the groupby and explore procedure
mapping_list_10 = icd10_umls_snomed.groupby('ICD_CODE')['SNOMED_CID'].apply(list)
mapping_list_10

In [ ]:
icd10['snomed'] = icd10['ICD9'].map(mapping_list_10)
icd10

In [ ]:
icd10_expanded = icd10.explode('snomed')
icd10_expanded

In [ ]:
# concatenate icd9 and icd10
icd_expanded = pd.concat([icd9_expanded, icd10_expanded])
icd_expanded

In [ ]:
# drop the helper columns
mimic_icd = icd_expanded.drop('icd_code', axis=1)
mimic_icd

In [ ]:
mimic_icd.reset_index(drop=True, inplace=True)
mimic_icd

In [ ]:
# remove duplicate rows
mimic_icd = mimic_icd.drop_duplicates(keep='first')
mimic_icd

In [ ]:
mimic_diagnoses = mimic_icd.rename(columns={'ICD9':'ICD'})

In [ ]:
mimic_icd.to_csv('mimic_diagnoses.csv', index=False)

In [ ]:
# the file is huge, we also create a shorter demo version where ICD9 column has unique values
mimic_diagnoses_small = mimic_diagnoses.drop_duplicates(subset=['subject_id', 'hadm_id', 'ICD'], keep='first')
mimic_diagnoses_small

In [ ]:
mimic_diagnoses_small.reset_index(drop=True, inplace=True)
mimic_diagnoses_small

In [ ]:
# drop SNOMED values that are nan
mimic_diagnoses_small = mimic_diagnoses_small.dropna(subset='snomed')
mimic_diagnoses_small

In [ ]:
mimic_diagnoses_small.to_csv('mimic_diagnoses_small.csv', index=False)

In [ ]:
service = pd.read_csv('services.csv')
service

In [ ]:
service = service[['hadm_id', 'curr_service']]
service

In [ ]:
service.to_csv('mimic_service.csv', index=False)